In [ ]:
!apt-get update
!apt-get install -y wget unzip
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb || apt-get -fy install

!pip install selenium webdriver-manager openpyxl

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException
from webdriver_manager.chrome import ChromeDriverManager
import time
import openpyxl

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://dl.google.com/linux/chrome-stable/deb stable InRelease [1,825 B]
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:6 https://dl.google.com/linux/chrome-stable/deb stable/main amd64 Packages [1,210 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Hit:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:14 http://archive.ubuntu.com/u

In [ ]:
options = Options()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

options.binary_location = "/usr/bin/google-chrome"

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

base_url = "https://realt.by/sale/flats/"
driver.get(base_url)

links = set()
page = 1

try:
    while len(links) < 500:
        time.sleep(2)

        elements = driver.find_elements(By.XPATH, '//a[contains(@href, "/sale-flats/object/")]')
        for el in elements:
            href = el.get_attribute("href")
            if href and "/sale-flats/object/" in href:
                links.add(href)

        try:
            next_button = driver.find_element(By.XPATH, '//a[@aria-label="Следующая страница"]')
            driver.execute_script("arguments[0].scrollIntoView(true);", next_button)
            time.sleep(1)
            driver.execute_script("arguments[0].click();", next_button)
            page += 1
        except NoSuchElementException:
            print("Следующая страница не найдена")
            break
        except ElementClickInterceptedException:
            print("Обнаружено перекрывающее окно, пытаемся закрыть...")
            try:
                close_button = driver.find_element(By.XPATH, '//button[@aria-label="Закрыть"]')
                close_button.click()
                time.sleep(1)
                driver.execute_script("arguments[0].click();", next_button)
                page += 1
            except:
                print("Не удалось закрыть окно, используем JavaScript клик")
                driver.execute_script("arguments[0].click();", next_button)
                page += 1
finally:
    driver.quit()

In [ ]:
import re
import pandas as pd
fields = {
    "Количество комнат": "",
    "Площадь общая": "",
    "Площадь жилая": "",
    "Год постройки": "",
    "Этаж / этажность": ""
}

data = []
i = 0
for link in links:
    i += 1
    driver = webdriver.Chrome(options=options)
    try:
        driver.get(link)
        time.sleep(1)

        try:
            area = driver.find_element(
                By.XPATH,
                '//ul[contains(@class,"w-full")]//span[text()="Область"]/ancestor::li//div[@class="w-1/2"]/a'
            ).text.strip()
        except NoSuchElementException:
            area = "Не указано"
        try:
            city = driver.find_element(
                By.XPATH,
                '//ul[contains(@class,"w-full")]//span[text()="Населенный пункт"]/ancestor::li//div[@class="w-1/2"]/a'
            ).text.strip()
        except NoSuchElementException:
            city = "Не указано"

        try:
            coord_element = driver.find_element(By.XPATH, "//li[div//span[contains(text(),'Координаты')]]//p")
            coord_text = coord_element.text.strip()
        except NoSuchElementException:
            coord_text = "Не найдено"
        except Exception as e:
            print(f"Ошибка при парсинге координат на {link}: {e}")
            coord_text = "Ошибка"

        try:
            try:
                show_more_button = driver.find_element(
                    By.XPATH,
                    '//button[.//span[contains(text(), "Показать больше")]]'
                )
                driver.execute_script("arguments[0].click();", show_more_button)
                time.sleep(1)
            except:
                pass

            description_div = driver.find_element(
                By.XPATH,
                '//section[.//h3[text()="Описание"]]//div[contains(@class, "description_wrapper")]//div'
            )

            description_html = description_div.get_attribute('innerHTML')

            from bs4 import BeautifulSoup
            soup = BeautifulSoup(description_html, 'html.parser')
            description = soup.get_text(separator=' ', strip=True)

        except NoSuchElementException:
            try:
                description_section = driver.find_element(
                    By.XPATH,
                    '//section[.//h3[text()="Описание"]]'
                )
                description = description_section.text.strip()
            except:
                description = "Описание не найдено"
        except Exception as e:
            description = f"Ошибка при парсинге описания: {e}"

        if description and len(description) < 100:
            try:
                paragraphs = driver.find_elements(
                    By.XPATH,
                    '//section[.//h3[text()="Описание"]]//p'
                )
                full_text = ""
                for p in paragraphs:
                    full_text += p.text.strip() + " "
                if full_text.strip():
                    description = full_text.strip()
            except:
                pass

        result = {
            "Ссылка": link,
            "Область": area,
            "Населенный пункт": city,
            "Координаты": coord_text,
            "Описание": description.strip() if isinstance(description, str) else str(description)
        }

        for key in fields:
            try:
                value = driver.find_element(
                    By.XPATH,
                    f'//ul[contains(@class,"w-full")]//span[text()="{key}"]/ancestor::li//div[@class="w-1/2"]/p'
                ).text.strip()
            except NoSuchElementException:
                value = "Не указано"

            if key == "Этаж / этажность":
                value = re.match(r'(\d)', value)
                if value:
                    value = value.group(1)
                else:
                    value = "Не указано"

            if key in ["Площадь общая", "Площадь жилая"]:
                value = re.sub(r'м²', '', value)

            result[key] = value

        try:
            price_usd_raw = driver.execute_script("""
                let spans = document.querySelectorAll('span');
                for (let span of spans) {
                    if (span.innerText.includes('≈') && span.innerText.includes('$')) {
                        return span.innerText;
                    }
                }
                return null;
            """)
            if price_usd_raw:
                match = re.search(r"≈\s*([\d\s\u00A0]+)\s*\$", price_usd_raw)
                if match:
                    price_usd = match.group(1).replace("\u00A0", "").replace(" ", "")
                else:
                    price_usd = "Не найдено"
            else:
                price_usd = "Не найдено"
        except Exception as e:
            price_usd = f"Ошибка: {e}"

        result["Цена в $"] = price_usd
        data.append(result)

        desc_len = len(result["Описание"]) if result["Описание"] else 0

    except Exception as e:
        print(f"Ошибка на {link}: {e}")
    finally:
        driver.quit()

df = pd.DataFrame(data)

df.rename(columns={"Этаж / этажность": "Этаж"}, inplace=True)

df.to_excel("flats_belarus.xlsx", index=False)
print("Данные сохранены в flats_belarus.xlsx")

Данные сохранены в flats_belarus.xlsx
